In [ ]:
# STEP 1: Mount Drive (Optional, for saving models)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# STEP 2: Setup Kaggle API and Download CADI-AI Dataset
!pip install -q kaggle

# Upload kaggle.json
from google.colab import files
files.upload()  # Upload your kaggle.json here

# Configure kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and unzip the CADI-AI dataset
!kaggle datasets download -d karaagroaiprojects/cadi-ai -p /content/Data --unzip


In [ ]:
import os
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# Fire Module from SqueezeNet
class Fire(nn.Module):
    def __init__(self, in_channels, squeeze_channels, expand_channels):
        super().__init__()
        self.squeeze = nn.Conv2d(in_channels, squeeze_channels, kernel_size=1)
        self.expand1 = nn.Conv2d(squeeze_channels, expand_channels, kernel_size=1)
        self.expand3 = nn.Conv2d(squeeze_channels, expand_channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = torch.relu(self.squeeze(x))
        return torch.relu(torch.cat([
            self.expand1(x), self.expand3(x)
        ], dim=1))

# Shuffle Block
class ShuffleBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, 1)
        self.depthwise = nn.Conv2d(in_channels, in_channels, 3, stride=1, padding=1, groups=in_channels)
        self.conv2 = nn.Conv2d(in_channels, in_channels, 1)

    def channel_shuffle(self, x):
        b, c, h, w = x.size()
        x = x.view(b, 2, c // 2, h, w)
        x = x.permute(0, 2, 1, 3, 4).contiguous()
        return x.view(b, c, h, w)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.depthwise(x))
        x = torch.relu(self.conv2(x))
        return self.channel_shuffle(x)

# FusionNet Model
class FusionNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        mobilenet = models.mobilenet_v3_small(pretrained=True)
        self.base = nn.Sequential(*list(mobilenet.features.children())[:6])  # shallow features

        self.fire = Fire(40, 16, 32)            # Fire module
        self.shuffle = ShuffleBlock(64)         # Shuffle block
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.base(x)              # MobileNetV3
        x = self.fire(x)              # Fire
        x = self.shuffle(x)           # Shuffle
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FusionNet(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(10):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/10], Loss: {total_loss:.4f}")


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Validation Accuracy: {accuracy:.2f}%")
